# RF-DETR Keypoint vs YOLO26-Pose: An Independent, Visual Head-to-Head

Roboflow released **RF-DETR Keypoint** (preview, Apache 2.0) in June 2026 and reports that it edges out **YOLO26x-pose** on COCO keypoints. This notebook checks that claim the practical way: both models see the same pictures and clips, and we look at the skeletons they draw.

**What you will do**
- Run both pretrained COCO pose models on 27 openly licensed photos and 6 short clips, with a fresh set of scenes in every section: field and racket sports, dance, gymnastics, martial arts, crowds, night scenes, silhouettes, aerial views and tiny far-away people
- Draw both skeletons with the same colors, RF-DETR on the left and YOLO26 on the right
- Look at what each model knows about its own uncertainty, and how NMS-free YOLO26 compares with its default NMS path
- Build an automatic failure gallery that shows where each model misses
- Fine-tune **both** models on the same small animal dataset (Ultralytics Tiger-Pose, no API key) with the same epoch budget, and compare before and after

**Runtime:** Colab **T4 GPU** (Runtime > Change runtime type > T4 GPU), about an hour end to end, roughly 47 minutes of it the two fine-tunes. Everything, including the dataset, downloads inside the notebook. Every figure is saved to `outputs/` and zipped at the end.

LearnOpenCV is independent from Roboflow and Ultralytics. Neither company reviewed this notebook.

## 1. Install pinned versions

We pin the exact releases this notebook was written against. `rfdetr[train,augment]` brings the PyTorch Lightning training stack and the Albumentations resize path that RF-DETR's `predict()` expects by default; `ultralytics` provides YOLO26-pose. The cell output is hidden to keep the notebook readable.

In [ ]:
%%capture
!pip install -q "rfdetr[train,augment]==1.12.1" "ultralytics==8.4.213" "supervision==0.31.7" "imageio-ffmpeg==0.6.0"

## 2. Imports, quiet logs and a GPU check

Both libraries are chatty, so we silence Python warnings and raise the log level of every logger that would otherwise flood the output (RF-DETR logs under the name `rf-detr`). We also create the `outputs/` folder where each figure is written, and confirm that a CUDA GPU is attached.

In [ ]:
import os, warnings, logging, time, glob, shutil, subprocess, zipfile, urllib.request, urllib.error
warnings.filterwarnings("ignore")
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

NOISY = ["ultralytics", "rf-detr", "rfdetr", "pytorch_lightning", "lightning", "lightning.pytorch",
         "lightning_fabric", "lightning.fabric", "transformers", "torch", "PIL", "matplotlib", "albumentations"]
def quiet_logs():
    for name in NOISY:
        logging.getLogger(name).setLevel(logging.ERROR)
quiet_logs()

import numpy as np, cv2, torch, imageio
import matplotlib.pyplot as plt
from IPython.display import display, Image as IPImage
from IPython.utils.capture import capture_output
import rfdetr, ultralytics, supervision as sv
quiet_logs()

print("rfdetr      ", getattr(rfdetr, "__version__", "installed"))
print("ultralytics ", ultralytics.__version__)
print("supervision ", sv.__version__)
print("torch       ", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

OUT = "outputs"
os.makedirs(OUT, exist_ok=True)
os.makedirs("assets", exist_ok=True)

## 3. Download the test photos and clips

All media comes from Wikimedia Commons under CC0, CC BY or CC BY-SA licenses (full credits are in `assets/CREDITS.md` in the code download). Each photo and clip is used in exactly one section, so every section shows new scenes. The six videos are cut to short clips with `ffmpeg` right after download so they match the clips used in the post. If you already have the `assets/` folder from the code download, nothing is fetched; you can also point `ASSET_BASE` at any folder URL that hosts the same files.

In [ ]:
ASSET_BASE = "https://raw.githubusercontent.com/spmallick/learnopencv/master/RF-DETR-Keypoint-vs-YOLO26-Pose/assets/"   # or a local folder path / another mirror

IMAGES = [
    # Sections 6 to 10: the main gallery
    "football_tackle.jpg", "basketball_layup.jpg", "tennis_serve.jpg", "table_tennis.jpg",
    "skateboard_air.jpg", "ballet_leap.jpg", "gymnast_beam.jpg", "yoga_group.jpg",
    "breakdance_headfreeze.jpg", "judo_throw.jpg", "crowded_street.jpg", "marathon_start.jpg",
    "night_market.jpg", "cyclist_silhouette.jpg", "beach_far_people.jpg",
    # Section 12: uncertainty
    "climber_wall.jpg", "workers_scaffolding.jpg",
    # Section 13: NMS vs NMS-free
    "shibuya_crossing.jpg", "festival_crowd.jpg",
    # Section 14: resolution knob
    "kids_street_football.jpg",
    # Section 15: failure batch
    "swimming_butterfly.jpg", "horse_show_jumping.jpg", "rock_band_stage.jpg",
    "wheelchair_basketball.jpg", "rugby_match.jpg", "beach_volleyball.jpg",
    # Section 19: out-of-distribution tiger for the fine-tune
    "tiger_wild.jpg",
]
VIDEOS = [
    "skateboarding_crowd.mp4", "street_dance_crowd.mp4", "capoeira_roda.mp4",
    "crosswalk_singapore.mp4", "boxing_training.mp4", "breakdance_airflare.mp4",
]

def fetch(name, tries=5):
    dst = os.path.join("assets", name)
    if os.path.exists(dst):
        return
    url = ASSET_BASE + name
    for i in range(tries):
        try:
            with urllib.request.urlopen(urllib.request.Request(url), timeout=120) as r, open(dst, "wb") as f:
                shutil.copyfileobj(r, f)
            return
        except urllib.error.HTTPError as e:
            print(f"  HTTP {e.code} for {name}, retrying...")
            time.sleep(5 * (i + 1))
    raise RuntimeError(f"could not download {url}")

for name in IMAGES + VIDEOS:
    fetch(name)

print(len(IMAGES), "photos and", len(VIDEOS), "clips ready,",
      round(sum(os.path.getsize(os.path.join("assets", f)) for f in os.listdir("assets")) / 1e6, 1), "MB")

## 4. Load both pretrained COCO pose models

`RFDETRKeypointPreview()` downloads Roboflow's COCO checkpoint (17 body keypoints, 576 x 576 input) on first use. Calling `model.inference(compile=False)` then returns RF-DETR's optimized, inference-only predictor, which is the path the library recommends for prediction (plain `model.predict()` works too, but logs a warning that it is not optimized). For YOLO26 we load the largest pose model, `yolo26x-pose.pt`, because that is the model Roboflow compares against.

One detail matters for fairness: in current Ultralytics releases YOLO26 runs its one-to-many head **with NMS by default**, and you opt into the end-to-end, NMS-free head with `nms=False`. RF-DETR has no NMS at all, so every comparison below runs YOLO26 with `nms=False`, and Section 13 shows the default path once.

In [ ]:
from rfdetr import RFDETRKeypointPreview
from ultralytics import YOLO

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
rf = RFDETRKeypointPreview()            # COCO person keypoints, 576 x 576 (source model, any input size)
try:
    RF_PRED = rf.inference(compile=False)   # optimized inference-only predictor at 576 x 576
except Exception as e:
    print("optimized predictor unavailable, using model.predict():", e)
    RF_PRED = rf
yolo_e2e = YOLO("yolo26x-pose.pt")      # we always call this one with nms=False
yolo_nms = YOLO("yolo26x-pose.pt")      # separate instance for the default (NMS) path
quiet_logs()

# Each vendor's documented defaults:
RF_THR = 0.5        # RF-DETR docs: predict(..., threshold=0.5)
YOLO_CONF = 0.25    # Ultralytics predict default conf
RF_KPT_THR = 0.1    # RF-DETR keypoint cookbook hides joints below 0.1 findability
YOLO_KPT_THR = 0.25 # Ultralytics keypoint plotting default (conf_thres=0.25)
print("models ready on", DEVICE, "| RF-DETR predictor:", type(RF_PRED).__name__)

## 5. One output format and one drawing style for both models

The two libraries return different objects: RF-DETR gives a `supervision.KeyPoints` with per-joint `keypoint_confidence` and a 2 x 2 covariance per joint, while Ultralytics gives a `Results` object with `keypoints.xy` and `keypoints.conf`. The helpers below convert both into the same small dictionary and draw them with identical colors (orange for the left side of the body, blue for the right, green for the center line), so any difference you see comes from the model and not from the plotting code.

In [ ]:
COCO_EDGES = [(15, 13), (13, 11), (16, 14), (14, 12), (11, 12), (5, 11), (6, 12), (5, 6), (5, 7),
              (6, 8), (7, 9), (8, 10), (1, 2), (0, 1), (0, 2), (1, 3), (2, 4), (3, 5), (4, 6)]
COCO_LEFT, COCO_RIGHT = {1, 3, 5, 7, 9, 11, 13, 15}, {2, 4, 6, 8, 10, 12, 14, 16}
C_LEFT, C_RIGHT, C_MID = (0, 140, 255), (255, 120, 0), (60, 200, 60)   # BGR
RF_COLOR, YOLO_COLOR = (140, 50, 120), (150, 90, 0)                     # title bars
TIMES = {"rf": [], "yolo": []}   # printed in the notebook only

def _empty(k=17):
    return dict(xy=np.zeros((0, k, 2)), kconf=np.zeros((0, k)), score=np.zeros(0),
                boxes=np.zeros((0, 4)), cov=None)

def _boxes_from_kpts(xy):
    return np.array([[p[:, 0].min(), p[:, 1].min(), p[:, 0].max(), p[:, 1].max()] for p in xy]).reshape(-1, 4)

def _sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def pred_rf(img_bgr, model=None, thr=RF_THR, shape=None, person_only=True, k=17):
    model = RF_PRED if model is None else model
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    kw = {} if shape is None else {"shape": shape}
    _sync(); t0 = time.perf_counter()
    kp = model.predict(rgb, threshold=thr, include_source_image=False, **kw)
    _sync(); TIMES["rf"].append(time.perf_counter() - t0)
    xy = np.asarray(kp.xy, dtype=float)
    if xy.ndim != 3 or len(xy) == 0:
        return _empty(k)
    n, kk = xy.shape[:2]
    kc = getattr(kp, "keypoint_confidence", None)
    kc = np.ones((n, kk)) if kc is None else np.asarray(kc, dtype=float)
    dc = getattr(kp, "detection_confidence", None)
    dc = np.ones(n) if dc is None else np.asarray(dc, dtype=float)
    data = getattr(kp, "data", None) or {}
    boxes = np.asarray(data["xyxy"], dtype=float) if "xyxy" in data else _boxes_from_kpts(xy)
    cov = np.asarray(data["covariance"], dtype=float) if "covariance" in data else None
    keep = np.ones(n, bool)
    if person_only and "class_name" in data:
        is_person = np.array([str(c) == "person" for c in data["class_name"]])
        if is_person.any():
            keep = is_person
    return dict(xy=xy[keep], kconf=kc[keep], score=dc[keep], boxes=boxes[keep],
                cov=None if cov is None else cov[keep])

def pred_yolo(img_bgr, model=None, conf=YOLO_CONF, nms_free=True, imgsz=640, k=17):
    if model is None:
        model = yolo_e2e if nms_free else yolo_nms
    kw = {"nms": False} if nms_free else {}
    _sync(); t0 = time.perf_counter()
    r = model.predict(img_bgr, conf=conf, imgsz=imgsz, verbose=False, **kw)[0]
    _sync(); TIMES["yolo"].append(time.perf_counter() - t0)
    if r.keypoints is None or r.boxes is None or len(r.boxes) == 0:
        return _empty(k)
    xy = r.keypoints.xy.cpu().numpy().astype(float)
    kc = r.keypoints.conf
    kc = np.ones(xy.shape[:2]) if kc is None else kc.cpu().numpy().astype(float)
    return dict(xy=xy, kconf=kc, score=r.boxes.conf.cpu().numpy(), boxes=r.boxes.xyxy.cpu().numpy(), cov=None)

def draw_pose(img, p, kthr, edges=COCO_EDGES, left=COCO_LEFT, right=COCO_RIGHT, boxes=True):
    out = img.copy()
    H, W = out.shape[:2]
    t = max(2, int(round(min(H, W) / 320)))
    for i in range(len(p["xy"])):
        pts, kc = p["xy"][i], p["kconf"][i]
        ok = (kc >= kthr) & ((pts[:, 0] > 0) | (pts[:, 1] > 0))
        if boxes:
            x1, y1, x2, y2 = p["boxes"][i].astype(int)
            cv2.rectangle(out, (x1, y1), (x2, y2), (230, 230, 230), 1, cv2.LINE_AA)
        for a, b in edges:
            if ok[a] and ok[b]:
                c = C_LEFT if (a in left and b in left) else C_RIGHT if (a in right and b in right) else C_MID
                cv2.line(out, tuple(pts[a].astype(int)), tuple(pts[b].astype(int)), c, t, cv2.LINE_AA)
        for j in np.where(ok)[0]:
            c = C_LEFT if j in left else C_RIGHT if j in right else C_MID
            cv2.circle(out, tuple(pts[j].astype(int)), t + 1, c, -1, cv2.LINE_AA)
            cv2.circle(out, tuple(pts[j].astype(int)), t + 1, (255, 255, 255), 1, cv2.LINE_AA)
    return out

def titled(img, text, color):
    H, W = img.shape[:2]
    bar = int(np.clip(H // 20, 30, 56))
    scale = bar / 42
    (tw, _), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, max(1, int(scale * 2)))
    if tw > W - 20:                       # shrink long titles so they never run off the panel
        scale *= (W - 20) / tw
    out = np.full((H + bar, W, 3), 255, np.uint8)
    out[:bar] = color
    out[bar:] = img
    cv2.putText(out, text, (10, int(bar * 0.72)), cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255),
                max(1, int(scale * 2)), cv2.LINE_AA)
    return out

def hstack(images, gap=8):
    h = max(i.shape[0] for i in images)
    rs = [cv2.resize(i, (int(i.shape[1] * h / i.shape[0]), h)) for i in images]
    sep = np.full((h, gap, 3), 255, np.uint8)
    out = rs[0]
    for r in rs[1:]:
        out = np.hstack([out, sep, r])
    return out

def vstack(images, gap=8):
    w = max(i.shape[1] for i in images)
    rs = [cv2.resize(i, (w, int(i.shape[0] * w / i.shape[1]))) for i in images]
    sep = np.full((gap, w, 3), 255, np.uint8)
    out = rs[0]
    for r in rs[1:]:
        out = np.vstack([out, sep, r])
    return out

def fit(img, W, H):
    # letterbox onto a white W x H canvas without distorting the aspect ratio
    s = min(W / img.shape[1], H / img.shape[0])
    r = cv2.resize(img, (max(1, int(img.shape[1] * s)), max(1, int(img.shape[0] * s))), interpolation=cv2.INTER_AREA)
    out = np.full((H, W, 3), 255, np.uint8)
    y, x = (H - r.shape[0]) // 2, (W - r.shape[1]) // 2
    out[y:y + r.shape[0], x:x + r.shape[1]] = r
    return out

def limit_width(img, max_w=2400):
    if img.shape[1] <= max_w:
        return img
    s = max_w / img.shape[1]
    return cv2.resize(img, (max_w, int(img.shape[0] * s)), interpolation=cv2.INTER_AREA)

def save_show(img_bgr, fname, width=16):
    img_bgr = limit_width(img_bgr)
    path = os.path.join(OUT, fname)
    cv2.imwrite(path, img_bgr, [cv2.IMWRITE_JPEG_QUALITY, 90])
    h, w = img_bgr.shape[:2]
    plt.figure(figsize=(width, width * h / w))
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.axis("off"); plt.tight_layout(); plt.show()
    print("saved", path)

def run_both(name):
    img = cv2.imread(os.path.join("assets", name))
    p_rf, p_y = pred_rf(img), pred_yolo(img)
    return dict(img=img, p_rf=p_rf, p_y=p_y,
                a_rf=draw_pose(img, p_rf, RF_KPT_THR), a_y=draw_pose(img, p_y, YOLO_KPT_THR))

def side_by_side(g):
    return hstack([titled(g["a_rf"], f"RF-DETR Keypoint | people: {len(g['p_rf']['xy'])}", RF_COLOR),
                   titled(g["a_y"], f"YOLO26x-pose (NMS-free) | people: {len(g['p_y']['xy'])}", YOLO_COLOR)])

GALLERY = {}   # name -> dict(img, p_rf, p_y, a_rf, a_y) for the main gallery (Sections 6 to 10)

def compare(name, fname):
    g = run_both(name)
    GALLERY[name] = g
    save_show(side_by_side(g), fname)
    return g["p_rf"], g["p_y"]

# warm-up on a blank frame so the first timed call is not dominated by CUDA initialization
_w = np.full((480, 640, 3), 127, np.uint8); pred_rf(_w); pred_yolo(_w); pred_yolo(_w, nms_free=False)
TIMES = {"rf": [], "yolo": []}
print("helpers ready")

## 6. First look: a football tackle

A tackle is a good opening test: two players overlap, limbs cross, and one player is falling. We run both models once and print the raw shapes they return, then draw the side-by-side panel that the next sections reuse as a format (each with new photos). Watch the crossed legs and the arm of the player in red.

In [ ]:
p_rf, p_y = compare("football_tackle.jpg", "01_football_tackle_side_by_side.jpg")
print("RF-DETR : xy", p_rf["xy"].shape, "| keypoint conf", p_rf["kconf"].shape,
      "| covariance", None if p_rf["cov"] is None else p_rf["cov"].shape)
print("YOLO26  : xy", p_y["xy"].shape, "| keypoint conf", p_y["kconf"].shape)

## 7. Sports: basketball, tennis, table tennis and skateboarding

Sports frames are where pose models earn their keep: fast limbs, stretched poses and players partly hidden by each other or by equipment. The basketball layup packs several bodies into a small area, the tennis serve has an arm fully above the head, the outdoor table tennis shot has spectators in the background, and the skateboarder is airborne with his body folded.

In [ ]:
for i, name in enumerate(["basketball_layup.jpg", "tennis_serve.jpg", "table_tennis.jpg", "skateboard_air.jpg"]):
    compare(name, f"02_sports_{i + 1}_{name.replace('.jpg', '')}_side_by_side.jpg")

## 8. Dance, gymnastics, yoga and martial arts: unusual poses

COCO is dominated by people standing, walking and sitting, so poses far from upright are the real test of generalization. Here we have a ballet leap, a gymnast on the beam, a yoga class in a twisted side angle, a breakdancer frozen upside down on his head, and a judo throw where two bodies tangle on the mat. Upside-down people are especially telling, because a model that learned "head above hips" as a shortcut will flip or drop joints.

In [ ]:
for i, name in enumerate(["ballet_leap.jpg", "gymnast_beam.jpg", "yoga_group.jpg",
                          "breakdance_headfreeze.jpg", "judo_throw.jpg"]):
    compare(name, f"03_poses_{i + 1}_{name.replace('.jpg', '')}_side_by_side.jpg")

## 9. Crowds, motion blur and low light

Crowded scenes stress a different part of the model: telling instances apart. A pedestrian mall has dozens of people at different scales, the marathon start has runners smeared by motion blur, and the night market is a dense crowd under dusk lighting. For a query-based model like RF-DETR the cap is the number of object queries (100 for this checkpoint), while YOLO26's end-to-end head keeps up to 300 candidates.

In [ ]:
for i, name in enumerate(["crowded_street.jpg", "marathon_start.jpg", "night_market.jpg"]):
    compare(name, f"04_crowds_{i + 1}_{name.replace('.jpg', '')}_side_by_side.jpg")

## 10. Silhouettes and tiny far-away people

Two images push visibility to the limit. The cyclist is a dark silhouette against a bright sky, so there is shape but almost no texture. The beach shot has a handful of people near the waterline who are only a few dozen pixels tall at 1280 pixels wide, and both models shrink the image further (576 for RF-DETR, 640 for YOLO26) before they see it. After the panels we print how many people each model found in every gallery photo so far.

In [ ]:
for i, name in enumerate(["cyclist_silhouette.jpg", "beach_far_people.jpg"]):
    compare(name, f"05_visibility_{i + 1}_{name.replace('.jpg', '')}_side_by_side.jpg")

print("People found per gallery photo (RF-DETR / YOLO26):")
for name, g in GALLERY.items():
    print(f"  {name:28s} {len(g['p_rf']['xy']):3d} / {len(g['p_y']['xy']):3d}")

## 11. How each model scores its joints

Before looking at uncertainty in pictures, it helps to see the raw numbers. This histogram pools every per-keypoint score from the 15 gallery photos above (no new images, just their predictions). RF-DETR's `keypoint_confidence` is a "findability" score, while YOLO26's `keypoints.conf` is a visibility-style score; the dashed line shows each vendor's display threshold, which is why we never force one shared threshold on both.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for a, key, title, thr in [(ax[0], "p_rf", "RF-DETR keypoint_confidence", RF_KPT_THR),
                           (ax[1], "p_y", "YOLO26 keypoints.conf", YOLO_KPT_THR)]:
    vals = np.concatenate([g[key]["kconf"].ravel() for g in GALLERY.values()] or [np.zeros(1)])
    a.hist(vals, bins=40, color="#7a3c8c" if key == "p_rf" else "#005a96")
    a.axvline(thr, color="k", ls="--", label=f"display threshold {thr}")
    a.set_title(title); a.set_xlabel("per-keypoint score"); a.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT, "06_keypoint_confidence_histograms.png"), dpi=120); plt.show()

## 12. What each model knows about its own uncertainty

RF-DETR predicts a full 2 x 2 covariance for every joint, which we draw as a 1-sigma ellipse (the same convention as supervision's `VertexEllipseAnnotator`, with a size cap so one wild joint cannot cover the frame): a small ellipse means a confident location, a long one means the model is unsure along that direction. YOLO26's pose head is trained with Residual Log-Likelihood Estimation, yet at inference it exposes only one score per joint, so we draw its joints with a circle that grows as that score drops. To keep the picture readable we draw these markers only on the main subjects (people taller than a fifth of the frame). Two new scenes: a rock climber whose own body hides some joints, and construction workers partly behind scaffolding.

In [ ]:
def main_people(p, H, frac=0.2):
    return [i for i in range(len(p["boxes"])) if (p["boxes"][i][3] - p["boxes"][i][1]) >= frac * H]

def draw_uncertainty_rf(img, p, kthr=RF_KPT_THR, sigma=1.0):
    out = draw_pose(img, p, kthr, boxes=False)
    if p["cov"] is None:
        print("This checkpoint returned no covariance; showing skeleton only.")
        return out
    H, W = img.shape[:2]
    max_axis = max(12, int(0.06 * min(H, W)))
    for i in main_people(p, H):
        for j in range(p["xy"].shape[1]):
            if p["kconf"][i, j] < kthr:
                continue
            vals, vecs = np.linalg.eigh(p["cov"][i, j])
            vals = np.clip(vals, 1e-6, None)
            axes = (int(min(max_axis, sigma * np.sqrt(vals[1]))), int(min(max_axis, sigma * np.sqrt(vals[0]))))
            ang = float(np.degrees(np.arctan2(vecs[1, 1], vecs[0, 1])))
            cv2.ellipse(out, tuple(p["xy"][i, j].astype(int)), (max(axes[0], 2), max(axes[1], 2)), ang, 0, 360,
                        (0, 255, 255), 2, cv2.LINE_AA)
    return out

def draw_uncertainty_yolo(img, p, kthr=YOLO_KPT_THR):
    out = draw_pose(img, p, kthr, boxes=False)
    H, W = img.shape[:2]
    max_r = max(12, int(0.06 * min(H, W)))
    for i in main_people(p, H):
        for j in range(p["xy"].shape[1]):
            s = p["kconf"][i, j]
            if s < kthr:
                continue
            r = int(3 + (max_r - 3) * (1 - s))
            cv2.circle(out, tuple(p["xy"][i, j].astype(int)), r, (0, 255, 255), 2, cv2.LINE_AA)
    return out

for name in ["climber_wall.jpg", "workers_scaffolding.jpg"]:
    img = cv2.imread(os.path.join("assets", name))
    p_rf, p_y = pred_rf(img), pred_yolo(img)
    panel = hstack([titled(draw_uncertainty_rf(img, p_rf), "RF-DETR: 1-sigma covariance ellipses", RF_COLOR),
                    titled(draw_uncertainty_yolo(img, p_y), "YOLO26: circle grows as score drops", YOLO_COLOR)])
    save_show(panel, f"07_uncertainty_{name.replace('.jpg', '')}.jpg")

## 13. YOLO26 with and without NMS

YOLO26 was designed around an NMS-free, one-to-one head, but Ultralytics now routes `predict()` through the one-to-many head plus NMS unless you pass `nms=False`. In a dense crowd the two paths can disagree, because NMS suppresses overlapping boxes by IoU while the one-to-one head was trained to output one box per person directly. Two new crowds: Tokyo's Shibuya scramble crossing and a music festival audience. Left is the NMS-free head used everywhere else in this notebook, right is the library default.

In [ ]:
for name in ["shibuya_crossing.jpg", "festival_crowd.jpg"]:
    img = cv2.imread(os.path.join("assets", name))
    pe, pn = pred_yolo(img, nms_free=True), pred_yolo(img, nms_free=False)
    panel = hstack([titled(draw_pose(img, pe, YOLO_KPT_THR), f"YOLO26 nms=False | people: {len(pe['xy'])}", YOLO_COLOR),
                    titled(draw_pose(img, pn, YOLO_KPT_THR), f"YOLO26 default (NMS) | people: {len(pn['xy'])}", (90, 90, 90))])
    save_show(panel, f"08_yolo26_nms_vs_nmsfree_{name.replace('.jpg', '')}.jpg")

## 14. Resolution as a knob for small people seen from above

Both models let you change the inference resolution without retraining. Roboflow trained RF-DETR with weight-sharing architecture search, so the same checkpoint accepts other square sizes as long as they divide by 24 (patch size 12 times 2 windows); its launch post reports a stronger COCO result at 888. YOLO26 takes any `imgsz` that is a multiple of 32. The new test photo looks down from a high window on children playing football in a dusty street lot, so every person is small and seen from an unusual angle. The optimized RF-DETR predictor is fixed at 576, so for the 888 run we call the source model directly.

In [ ]:
img = cv2.imread("assets/kids_street_football.jpg")
runs = []
for label, fn in [("RF-DETR @576", lambda: pred_rf(img)),
                  ("RF-DETR @888", lambda: pred_rf(img, model=rf, shape=(888, 888))),
                  ("YOLO26 @640", lambda: pred_yolo(img, imgsz=640)),
                  ("YOLO26 @1280", lambda: pred_yolo(img, imgsz=1280))]:
    try:
        p = fn()
    except Exception as e:      # a custom shape is accepted by the API, but keep going if a release changes that
        print(label, "failed:", e); p = _empty()
    runs.append((label, p))

all_boxes = np.concatenate([p["boxes"] for _, p in runs if len(p["boxes"])] or [np.zeros((0, 4))])
H, W = img.shape[:2]
if len(all_boxes):
    x1, y1 = all_boxes[:, :2].min(0); x2, y2 = all_boxes[:, 2:].max(0)
    pad = 0.1 * max(x2 - x1, y2 - y1, 120)
    x1, y1, x2, y2 = int(max(0, x1 - pad)), int(max(0, y1 - pad)), int(min(W, x2 + pad)), int(min(H, y2 + pad))
else:
    x1, y1, x2, y2 = 0, 0, W, H
tiles = []
for label, p in runs:
    kthr = RF_KPT_THR if label.startswith("RF") else YOLO_KPT_THR
    crop = draw_pose(img, p, kthr)[y1:y2, x1:x2]
    crop = cv2.resize(crop, (900, int(900 * crop.shape[0] / crop.shape[1])), interpolation=cv2.INTER_CUBIC)
    tiles.append(titled(crop, f"{label} | people: {len(p['xy'])}", RF_COLOR if label.startswith("RF") else YOLO_COLOR))
save_show(vstack([hstack(tiles[:2]), hstack(tiles[2:])]), "09_resolution_knob_kids_street_football.jpg")
quiet_logs()

## 15. Failure gallery on a fresh batch: where each model misses

Instead of hand-picking failures, we let the data decide, and we use six photos that appear nowhere else in the notebook: a butterfly swimmer half under water, a show jumper with horse and rider, a black and white photo of a rock band on stage in front of a huge crowd, wheelchair basketball, a rugby scrum and beach volleyball. First we show the whole batch as RF-DETR | YOLO26 pairs. Then we match the two models' people by box overlap (IoU of at least 0.5): a person found by only one model goes into a "found only by" strip, and a matched person whose skeletons sit far apart goes into a "same person, different skeleton" strip. Each crop shows RF-DETR on the left and YOLO26 on the right for the same region. People under 6 percent of the image height are skipped.

In [ ]:
FAIL_BATCH = ["swimming_butterfly.jpg", "horse_show_jumping.jpg", "rock_band_stage.jpg",
              "wheelchair_basketball.jpg", "rugby_match.jpg", "beach_volleyball.jpg"]
BATCH = {name: run_both(name) for name in FAIL_BATCH}

tiles = []
for name, g in BATCH.items():
    pair = fit(hstack([g["a_rf"], g["a_y"]], gap=4), 964, 420)
    cv2.putText(pair, name[:-4], (8, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 3, cv2.LINE_AA)
    cv2.putText(pair, name[:-4], (8, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (20, 20, 20), 1, cv2.LINE_AA)
    tiles.append(pair)
if len(tiles) % 2:
    tiles.append(np.full_like(tiles[0], 255))
rows = [hstack(tiles[i:i + 2], gap=10) for i in range(0, len(tiles), 2)]
save_show(titled(vstack(rows, gap=10), "Each pair: RF-DETR Keypoint (left) | YOLO26x-pose NMS-free (right)", (60, 60, 60)),
          "10_failure_batch_overview.jpg", width=18)

## 16. Video: five clips, five kinds of motion

Still images hide one important property: stability over time. We run both models on sampled frames of five short clips, each a different scene: a skateboarding contest in front of a crowd, a street dancer in a ring of spectators, a capoeira game in a park, a busy scramble crosswalk, and a boxer working the pads in a gym. Each clip becomes a side-by-side MP4 and a looping GIF (RF-DETR left, YOLO26 right, the original video under each skeleton). We read each clip's real frame rate and sample frames so the output plays at 8 frames per second, covering about 7 seconds; the GIF is written at a smaller size and re-saved smaller still if it would exceed 4 MB.

In [ ]:
def save_gif(path, frames, fps, width=560, max_mb=4.0):
    scale = width / frames[0].shape[1]
    for _ in range(4):
        w = int(frames[0].shape[1] * scale) // 2 * 2
        h = int(frames[0].shape[0] * scale) // 2 * 2
        gif = [cv2.resize(f, (w, h), interpolation=cv2.INTER_AREA) for f in frames]
        try:
            imageio.mimsave(path, gif, fps=fps, loop=0)
        except TypeError:
            imageio.mimsave(path, gif, duration=1000 / fps, loop=0)
        if os.path.getsize(path) / 1e6 <= max_mb:
            break
        scale *= 0.85
    return os.path.getsize(path) / 1e6

def process_video(name, prefix, out_fps=8, max_frames=56, panel_w=560):
    cap = cv2.VideoCapture(os.path.join("assets", name))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames, track, raw, i, last = [], [], [], 0, -1
    while len(frames) < max_frames:
        ok, frame = cap.read()
        if not ok:
            break
        slot = int(i * out_fps / fps); i += 1
        if slot == last:
            continue
        last = slot
        p_rf, p_y = pred_rf(frame), pred_yolo(frame)
        track.append((p_rf, p_y)); raw.append(frame)
        a = titled(draw_pose(frame, p_rf, RF_KPT_THR), "RF-DETR Keypoint", RF_COLOR)
        b = titled(draw_pose(frame, p_y, YOLO_KPT_THR), "YOLO26x-pose", YOLO_COLOR)
        h = int(panel_w * a.shape[0] / a.shape[1])
        pair = hstack([cv2.resize(a, (panel_w, h)), cv2.resize(b, (panel_w, h))], gap=8)
        pair = pair[: pair.shape[0] // 16 * 16, : pair.shape[1] // 16 * 16]   # codec-friendly size
        frames.append(cv2.cvtColor(pair, cv2.COLOR_BGR2RGB))
    cap.release()
    stem = os.path.join(OUT, f"{prefix}_video_" + name.replace(".mp4", "") + "_side_by_side")
    imageio.mimsave(stem + ".mp4", frames, fps=out_fps, codec="libx264", quality=7)
    mb = save_gif(stem + ".gif", frames, out_fps)
    print(f"{name}: {fps:.0f} fps source, {total} frames -> {len(frames)} sampled | GIF {mb:.1f} MB")
    display(IPImage(filename=stem + ".gif"))
    return frames, track, raw

for name in ["skateboarding_crowd.mp4", "street_dance_crowd.mp4", "capoeira_roda.mp4",
             "crosswalk_singapore.mp4", "boxing_training.mp4"]:
    process_video(name, "14")

## 17. Temporal stability: tracing wrists and ankles through an airflare

A breakdance airflare spins the dancer on his hands with legs swinging overhead, which produces heavy motion blur and fast left-right swaps. This clip is used only here. We first write its side-by-side GIF, then follow the dancer from frame to frame (starting from the largest person and linking each frame to the nearest box of similar size) and draw the path of both wrists and both ankles. A path segment is drawn only when the joint moves less than 12 percent of the frame width between samples, so a long jump shows up as a gap instead of a line across the image. The second plot shows the left wrist height frame by frame.

In [ ]:
frames, track, raw = process_video("breakdance_airflare.mp4", "15")
JOINTS = {9: ("L wrist", C_LEFT), 10: ("R wrist", C_RIGHT), 15: ("L ankle", (0, 200, 255)), 16: ("R ankle", (255, 200, 0))}

def follow(track, which):
    pts, prev_c, prev_s = [], None, None
    for p_rf, p_y in track:
        p = p_rf if which == "rf" else p_y
        thr = RF_KPT_THR if which == "rf" else YOLO_KPT_THR
        if len(p["boxes"]) == 0:
            pts.append(None); continue
        c = (p["boxes"][:, :2] + p["boxes"][:, 2:]) / 2
        size = np.sqrt((p["boxes"][:, 2] - p["boxes"][:, 0]) * (p["boxes"][:, 3] - p["boxes"][:, 1]))
        if prev_c is None:
            k = int(np.argmax(size))
        else:
            cost = np.linalg.norm(c - prev_c, axis=1) / prev_s + np.abs(np.log(size / prev_s))
            k = int(np.argmin(cost))
            if cost[k] > 1.5:              # nobody close to where the dancer was: treat as a miss
                pts.append(None); continue
        prev_c, prev_s = c[k], size[k]
        pts.append({j: (p["xy"][k, j] if p["kconf"][k, j] >= thr else None) for j in JOINTS})
    return pts

base = raw[len(raw) // 2]
W = base.shape[1]
panels, curves = [], {}
for which, title, col in [("rf", "RF-DETR joint paths", RF_COLOR), ("yolo", "YOLO26 joint paths", YOLO_COLOR)]:
    pts = follow(track, which)
    canvas = (base * 0.45).astype(np.uint8)
    gaps = 0
    for j, (lab, c) in JOINTS.items():
        seq = [q[j] if q else None for q in pts]
        for a, b in zip(seq[:-1], seq[1:]):
            if a is None or b is None:
                continue
            if np.linalg.norm(a - b) > 0.12 * W:
                gaps += 1; continue
            cv2.line(canvas, tuple(a.astype(int)), tuple(b.astype(int)), c, 3, cv2.LINE_AA)
        curves[(which, j)] = [np.nan if s is None else s[1] for s in seq]
    print(f"{title}: {gaps} jumps larger than 12% of the frame width")
    panels.append(titled(canvas, title, col))
save_show(hstack(panels), "15_temporal_joint_paths_airflare.jpg")

plt.figure(figsize=(12, 3.5))
plt.plot(curves[("rf", 9)], "-o", ms=3, color="#7a3c8c", label="RF-DETR left wrist y")
plt.plot(curves[("yolo", 9)], "-o", ms=3, color="#005a96", label="YOLO26 left wrist y")
plt.gca().invert_yaxis(); plt.xlabel("sampled frame (8 per second)"); plt.ylabel("image y (px)")
plt.legend(); plt.title("Left wrist height over the airflare clip (gaps = joint hidden or not found)")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "16_temporal_left_wrist_curve.png"), dpi=120); plt.show()

For our own reference only (the post does not quote speed), here is the mean wall-clock time per image that this runtime measured for the two pretrained models in PyTorch, without TensorRT or any export.

In [ ]:
for k, v in TIMES.items():
    if v:
        print(f"{k:5s}: {1000 * np.mean(v):6.1f} ms mean over {len(v)} calls (PyTorch, this GPU)")

## 18. Fine-tuning data: Ultralytics Tiger-Pose

To compare fine-tuning fairly we need one dataset that both libraries read without conversion and without an API key. Ultralytics' Tiger-Pose has 210 training and 53 validation frames of a tiger, each labeled with 12 keypoints (nose, head, withers, tail base and three points per leg) in the YOLO pose format. RF-DETR reads the same folder with `dataset_file="yolo"`, picking up `kpt_shape`, `flip_idx` and keypoint names from `data.yaml`. The tiger frames come from a YouTube video and the annotations are AGPL-3.0, so we use them only inside this runtime.

In [ ]:
DS_ROOT = "/content/datasets" if os.path.isdir("/content") else os.path.abspath("datasets")
TIGER_DIR = os.path.join(DS_ROOT, "tiger-pose")
if not glob.glob(os.path.join(TIGER_DIR, "**", "images", "train"), recursive=True):
    os.makedirs(TIGER_DIR, exist_ok=True)
    zpath = os.path.join(DS_ROOT, "tiger-pose.zip")
    fetch("https://github.com/ultralytics/assets/releases/download/v0.0.0/tiger-pose.zip", zpath)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(TIGER_DIR)
    os.remove(zpath)
if not os.path.isdir(os.path.join(TIGER_DIR, "images", "train")):
    # the zip may or may not contain a top-level folder; find where images/ and labels/ landed
    hits = glob.glob(os.path.join(TIGER_DIR, "**", "images", "train"), recursive=True)
    TIGER_DIR = os.path.dirname(os.path.dirname(hits[0]))
print("Tiger-Pose root:", TIGER_DIR)

TIGER_KPTS = ["nose", "head", "withers", "tail_base", "right_hind_hock", "right_hind_paw", "left_hind_paw",
              "left_hind_hock", "right_front_wrist", "right_front_paw", "left_front_wrist", "left_front_paw"]
TIGER_YAML = os.path.join(TIGER_DIR, "data.yaml")
with open(TIGER_YAML, "w") as f:
    f.write(f"path: {TIGER_DIR}\ntrain: images/train\nval: images/val\n"
            "kpt_shape: [12, 2]\nflip_idx: [0, 1, 2, 3, 7, 6, 5, 4, 10, 11, 8, 9]\n"
            "names:\n  0: tiger\nkpt_names:\n  0: [" + ", ".join(TIGER_KPTS) + "]\n")

TIGER_EDGES = [(0, 1), (1, 2), (2, 3), (2, 8), (8, 9), (2, 10), (10, 11), (3, 4), (4, 5), (3, 7), (7, 6)]
TIGER_LEFT, TIGER_RIGHT = {6, 7, 10, 11}, {4, 5, 8, 9}
val_imgs = sorted(glob.glob(os.path.join(TIGER_DIR, "images", "val", "*")))
print(len(glob.glob(os.path.join(TIGER_DIR, "images", "train", "*"))), "train /", len(val_imgs), "val images")

def load_gt(img_path):
    lab = img_path.replace(os.sep + "images" + os.sep, os.sep + "labels" + os.sep).rsplit(".", 1)[0] + ".txt"
    img = cv2.imread(img_path); H, W = img.shape[:2]
    xy, boxes = [], []
    for line in open(lab).read().split("\n"):
        v = line.split()
        if len(v) < 5 + 24:
            continue
        v = np.array(v[1:], float)
        cx, cy, w, h = v[:4] * [W, H, W, H]
        boxes.append([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])
        xy.append(v[4:4 + 24].reshape(12, 2) * [W, H])
    xy = np.array(xy).reshape(-1, 12, 2)
    present = ((xy[..., 0] > 0) & (xy[..., 1] > 0)).astype(float)   # absent joints are stored as 0 or negative
    return img, dict(xy=xy, kconf=present, boxes=np.array(boxes).reshape(-1, 4), cov=None, score=np.ones(len(xy)))

# ground truth on three training frames; the validation frames are kept for the before/after test
train_imgs = sorted(glob.glob(os.path.join(TIGER_DIR, "images", "train", "*")))
gt_tiles = []
for pth in [train_imgs[i] for i in np.linspace(0, len(train_imgs) - 1, 3).astype(int)]:
    img, gt = load_gt(pth)
    gt_tiles.append(draw_pose(img, gt, 0.5, TIGER_EDGES, TIGER_LEFT, TIGER_RIGHT))
save_show(titled(hstack(gt_tiles), "Tiger-Pose ground truth (12 keypoints, training frames)", (60, 60, 60)),
          "17_tiger_ground_truth.jpg")

## 19. Before fine-tuning: COCO person models on a tiger

Both pretrained checkpoints only know the 17-point human skeleton, so on a tiger they should either find nothing or force a human skeleton onto the animal. This is the baseline that fine-tuning has to fix. We test three validation frames plus one tiger photo from Wikimedia Commons that is not part of the dataset, which we will use again after training as an out-of-distribution check (here the same inputs are the point of the comparison).

In [ ]:
TIGER_TEST = [val_imgs[i] for i in np.linspace(0, len(val_imgs) - 1, 3).astype(int)] + ["assets/tiger_wild.jpg"]
BEFORE = []
for pth in TIGER_TEST:
    img = cv2.imread(pth)
    p_rf = pred_rf(img, person_only=False); p_y = pred_yolo(img)
    BEFORE.append((draw_pose(img, p_rf, RF_KPT_THR), draw_pose(img, p_y, YOLO_KPT_THR)))
rows = [hstack([titled(a, "RF-DETR (COCO person)", RF_COLOR), titled(b, "YOLO26 (COCO person)", YOLO_COLOR)])
        for a, b in BEFORE]
save_show(vstack(rows), "18_tiger_before_finetune.jpg")

# free GPU memory before training
del rf, RF_PRED, yolo_e2e, yolo_nms
import gc; gc.collect(); torch.cuda.empty_cache()

## 20. Fine-tune YOLO26x-pose on Tiger-Pose

The fine-tuning budget is the same for both models: **20 epochs** each, starting from each vendor's COCO pose checkpoint, with each library's default augmentation and learning-rate recipe. On a T4 that is roughly 20 minutes for YOLO26x-pose and 28 minutes for RF-DETR, which keeps the whole notebook near an hour; it is a short schedule, so treat the results as "how quickly does each model adapt", not as fully converged models. The Ultralytics recipe is one call. Training logs go to a file instead of the notebook, and we print one summary line at the end with the trainer's own validation metrics.

In [ ]:
EPOCHS = 20

def run_quietly(fn, log_path):
    # keep progress bars and metric tables out of the notebook; everything is still saved to log_path
    quiet_logs()
    t0 = time.time()
    err = None
    with capture_output(display=True) as cap:
        try:
            fn()
        except Exception as e:
            err = e
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, "w") as f:
        f.write(cap.stdout + "\n" + cap.stderr)
    if err is not None:
        print("training failed, last log lines:\n" + "\n".join((cap.stdout + cap.stderr).splitlines()[-15:]))
        raise err
    return (time.time() - t0) / 60

y_ft = YOLO("yolo26x-pose.pt")
minutes = run_quietly(lambda: y_ft.train(data=TIGER_YAML, epochs=EPOCHS, imgsz=640, batch=8, workers=2, seed=0,
                                         project="runs", name="yolo26x_tiger", exist_ok=True, plots=False,
                                         verbose=False), "runs/logs/yolo26_train.log")
YOLO_BEST = str(getattr(y_ft.trainer, "best", "")) or sorted(glob.glob("runs/**/best.pt", recursive=True))[-1]
m = getattr(y_ft.trainer, "metrics", {}) or {}
print(f"YOLO26x-pose: {EPOCHS} epochs in {minutes:.1f} min | val pose mAP50-95 "
      f"{m.get('metrics/mAP50-95(P)', float('nan')):.3f}, mAP50 {m.get('metrics/mAP50(P)', float('nan')):.3f} | {YOLO_BEST}")
del y_ft; gc.collect(); torch.cuda.empty_cache()

## 21. Fine-tune RF-DETR Keypoint on the same folder

RF-DETR's `train()` reads the Ultralytics pose layout directly when you pass `dataset_file="yolo"`, and it infers the 12-keypoint schema, left-right flip pairs and OKS sigmas from `data.yaml`. We keep the 576 resolution and the library defaults, use a physical batch of 4 with 4 gradient-accumulation steps (an effective batch of 16 that fits a T4), and give it the same 20 epochs. Two settings are adjusted: best-checkpoint selection starts after epoch 5 instead of the keypoint default of 10, and the progress bar and metric tables are kept out of the notebook. If you hit an out-of-memory error, set `batch_size=2, grad_accum_steps=8`.

In [ ]:
import pandas as pd
from rfdetr import RFDETRKeypointPreview
RF_OUT = "runs/rfdetr_tiger"
rf_train = RFDETRKeypointPreview()
minutes = run_quietly(lambda: rf_train.train(dataset_dir=TIGER_DIR, dataset_file="yolo", epochs=EPOCHS,
                                             batch_size=4, grad_accum_steps=4, lr=1e-4, skip_best_epochs=5,
                                             output_dir=RF_OUT, tensorboard=False, progress_bar=None),
                      "runs/logs/rfdetr_train.log")
summary = []
csv_path = os.path.join(RF_OUT, "metrics.csv")
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    for col in [c for c in df.columns if c.endswith("keypoint_map_50_95") or c.endswith("keypoint_map_50")]:
        s = df[col].dropna()
        if len(s):
            summary.append(f"{col.split('/')[-1]} best {s.max():.3f}")
print(f"RF-DETR Keypoint: {EPOCHS} epochs in {minutes:.1f} min | " + (", ".join(summary) or "metrics.csv not found"))
RF_BEST = os.path.join(RF_OUT, "checkpoint_best_total.pth")

## 22. After fine-tuning: the same tiger frames

Now we load each model's best checkpoint and run it on the three validation frames and the unseen Wikimedia tiger from Section 19. Short fine-tunes produce less confident scores than a fully trained model, so we lower RF-DETR's detection threshold to 0.3 here and keep YOLO26 at its 0.25 default. The Tiger-Pose labels carry no visibility flag, so YOLO26 returns no per-joint score and every predicted joint is drawn.

In [ ]:
try:
    rf_ft = RFDETRKeypointPreview.from_checkpoint(RF_BEST)
    print("loaded", RF_BEST)
except Exception as e:
    print("using the in-memory trained model:", e)
    rf_ft = rf_train
try:
    rf_ft_pred = rf_ft.inference(compile=False)
except Exception:
    rf_ft_pred = rf_ft
quiet_logs()
y_ft = YOLO(YOLO_BEST)
FT_RF_THR = 0.3
AFTER = []
for pth in TIGER_TEST:
    img = cv2.imread(pth)
    p_rf = pred_rf(img, model=rf_ft_pred, thr=FT_RF_THR, person_only=False, k=12)
    p_y = pred_yolo(img, model=y_ft, nms_free=True, k=12)
    a = draw_pose(img, p_rf, RF_KPT_THR, TIGER_EDGES, TIGER_LEFT, TIGER_RIGHT)
    b = draw_pose(img, p_y, 0.0, TIGER_EDGES, TIGER_LEFT, TIGER_RIGHT)
    AFTER.append((a, b))
    print(os.path.basename(pth), "| RF-DETR tigers:", len(p_rf["xy"]), "| YOLO26 tigers:", len(p_y["xy"]))
rows = [hstack([titled(a, "RF-DETR fine-tuned", RF_COLOR), titled(b, "YOLO26 fine-tuned", YOLO_COLOR)]) for a, b in AFTER]
save_show(vstack(rows), "19_tiger_after_finetune.jpg")

grid = []
for (rb, yb), (ra, ya) in zip(BEFORE, AFTER):
    grid.append(hstack([titled(rb, "RF-DETR before", RF_COLOR), titled(ra, "RF-DETR after", RF_COLOR),
                        titled(yb, "YOLO26 before", YOLO_COLOR), titled(ya, "YOLO26 after", YOLO_COLOR)]))
save_show(vstack(grid), "20_tiger_before_after_grid.jpg", width=18)

## 23. Download every figure

All figures, MP4s and GIFs live in `outputs/`. The last cell zips the folder and, on Colab, starts the download. The two training logs are in `runs/logs/` if you want the full metric tables.

In [ ]:
for f in sorted(os.listdir(OUT)):
    print(f"{f:60s} {os.path.getsize(os.path.join(OUT, f)) / 1e6:6.2f} MB")
shutil.make_archive("rfdetr_vs_yolo26_pose_outputs", "zip", OUT)
try:
    from google.colab import files
    files.download("rfdetr_vs_yolo26_pose_outputs.zip")
except ImportError:
    print("Not on Colab: zip written to rfdetr_vs_yolo26_pose_outputs.zip")